In [1]:
from pathlib import Path
import pandas as pd
import pydicom

ROOT = Path(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)

train = pd.read_csv(ROOT / "train.csv")
series = pd.read_csv(ROOT / "train_series.csv")

# Check One Study and See its series

In [2]:
study_id = train["StudyInstanceUID"].iloc[0]

study_series = series[
    series["StudyInstanceUID"] == study_id
].copy()

display(study_series)

,StudyInstanceUID,SeriesInstanceUID,Fluid_Sensitive,Fat_Suppression,Anatomical_Plane
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.12343110195036213483...,1,1,Sagittal
1,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.13821229744997220641...,1,1,Axial
2,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.23084836536722595275...,0,0,Coronal
3,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.40734206102458723096...,1,1,Coronal
4,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.75714899997203615784...,0,0,Sagittal


# Check One Series then Locate that

In [3]:
series_id = study_series["SeriesInstanceUID"].iloc[0]

series_folder = ROOT / "train_series" / study_id / series_id
dicom_files = sorted(series_folder.glob("*.dcm"))

print("Series folder exists:", series_folder.exists())
print("Number of DICOM files:", len(dicom_files))

Series folder exists: True
Number of DICOM files: 22


# Read One File's Header

In [4]:
if not dicom_files:
    raise FileNotFoundError(f"No DICOM files found in {series_folder}")

dicom_path = dicom_files[0]

header = pydicom.dcmread(
    dicom_path,
    stop_before_pixels=True,
)

tags = [
    "StudyInstanceUID",
    "SeriesInstanceUID",
    "Modality",
    "SeriesDescription",
    "Rows",
    "Columns",
    "PixelSpacing",
    "SliceThickness",
    "ImagePositionPatient",
    "ImageOrientationPatient",
]

metadata = pd.DataFrame({
    "Field": tags,
    "Value": [str(getattr(header, tag, "Missing")) for tag in tags],
})

display(metadata)

,Field,Value
0,StudyInstanceUID,1.2.826.0.1.3680043.8.498.10004873229099053869...
1,SeriesInstanceUID,1.2.826.0.1.3680043.8.498.12343110195036213483...
2,Modality,MR
3,SeriesDescription,DP SPIR CS_SAG
4,Rows,512
5,Columns,512
6,PixelSpacing,"[0.330078125, 0.330078125]"
7,SliceThickness,3.4
8,ImagePositionPatient,"[-80.965252085414, -143.1194283448, 98.5467405..."
9,ImageOrientationPatient,"[-0.0334932804107, 0.99921125173568, 0.0213321..."


Do All files have the same dimensions?

In [5]:
records = []

for path in dicom_files:
    ds = pydicom.dcmread(path, stop_before_pixels=True)

    records.append({
        "Rows": getattr(ds, "Rows", None),
        "Columns": getattr(ds, "Columns", None),
        "PixelSpacing": str(getattr(ds, "PixelSpacing", None)),
        "SliceThickness": getattr(ds, "SliceThickness", None),
    })

slice_metadata = pd.DataFrame(records)

display(
    slice_metadata.value_counts(dropna=False)
    .reset_index(name="Number of files")
)

,Rows,Columns,PixelSpacing,SliceThickness,Number of files
0,512,512,"[0.330078125, 0.330078125]",3.4,22


Check, do they progress smoothly or jump around?

In [6]:
positions = []

for path in dicom_files:
    ds = pydicom.dcmread(path, stop_before_pixels=True)

    positions.append({
        "File": path.name,
        "InstanceNumber": getattr(ds, "InstanceNumber", None),
        "X_mm": float(ds.ImagePositionPatient[0]),
        "Y_mm": float(ds.ImagePositionPatient[1]),
        "Z_mm": float(ds.ImagePositionPatient[2]),
    })

positions = pd.DataFrame(positions)
display(positions)

,File,InstanceNumber,X_mm,Y_mm,Z_mm
0,1.2.826.0.1.3680043.8.498.10374285052733466977...,5,-80.965252,-143.119428,98.546741
1,1.2.826.0.1.3680043.8.498.10552300544162020167...,10,-104.085957,-143.865362,97.185281
2,1.2.826.0.1.3680043.8.498.10986732409717781858...,3,-71.716969,-142.821058,99.091325
3,1.2.826.0.1.3680043.8.498.11560456494251875828...,2,-67.092823,-142.671869,99.363618
4,1.2.826.0.1.3680043.8.498.12720245539969605839...,9,-99.461811,-143.716177,97.457573
5,1.2.826.0.1.3680043.8.498.13270416360166070027...,12,-113.334240,-144.163736,96.640697
6,1.2.826.0.1.3680043.8.498.16282623942824367613...,13,-117.958378,-144.312925,96.368405
7,1.2.826.0.1.3680043.8.498.23328002438949038536...,16,-131.830799,-144.760484,95.551529
8,1.2.826.0.1.3680043.8.498.25773224913662123081...,19,-145.703221,-145.208044,94.734653
9,1.2.826.0.1.3680043.8.498.33690734911291064161...,14,-122.582524,-144.462110,96.096113


In [7]:
ordered_positions = (
    positions.sort_values("InstanceNumber")
    .reset_index(drop=True)
)

display(ordered_positions)

,File,InstanceNumber,X_mm,Y_mm,Z_mm
0,1.2.826.0.1.3680043.8.498.81077736138294550629...,1,-62.468685,-142.522684,99.635909
1,1.2.826.0.1.3680043.8.498.11560456494251875828...,2,-67.092823,-142.671869,99.363618
2,1.2.826.0.1.3680043.8.498.10986732409717781858...,3,-71.716969,-142.821058,99.091325
3,1.2.826.0.1.3680043.8.498.93755789787124953238...,4,-76.341107,-142.970243,98.819034
4,1.2.826.0.1.3680043.8.498.10374285052733466977...,5,-80.965252,-143.119428,98.546741
5,1.2.826.0.1.3680043.8.498.79341978025976159266...,6,-85.589390,-143.268617,98.274449
6,1.2.826.0.1.3680043.8.498.46159856332493360402...,7,-90.213535,-143.417803,98.002157
7,1.2.826.0.1.3680043.8.498.56450775768487641304...,8,-94.837673,-143.566992,97.729865
8,1.2.826.0.1.3680043.8.498.12720245539969605839...,9,-99.461811,-143.716177,97.457573
9,1.2.826.0.1.3680043.8.498.10552300544162020167...,10,-104.085957,-143.865362,97.185281


In [8]:
import numpy as np

xyz = ordered_positions[["X_mm", "Y_mm", "Z_mm"]].to_numpy()

steps = np.diff(xyz, axis=0)
distances = np.linalg.norm(steps, axis=1)

print("Consecutive position distances (mm):")
print(np.round(distances, 3))

Consecutive position distances (mm):
[4.635 4.635 4.635 4.635 4.635 4.635 4.635 4.635 4.635 4.635 4.635 4.635
 4.635 4.635 4.635 4.635 4.635 4.635 4.635 4.635 4.635]


- The image positions progress regularly when sorted by InstanceNumber
- Slice thickness is 3.4 mm, while the distance between corresponding image origins is 4.635 mm.
- We still need to check orientation before calling that distance the perpendicular slice spacing.

In [9]:
# Read each file's orientation, keeping the same order as "positions".
orientations = np.array([
    pydicom.dcmread(
        series_folder / filename,
        stop_before_pixels=True,
    ).ImageOrientationPatient
    for filename in positions["File"]
], dtype=float)

# Check whether all slices have the same orientation.
same_orientation = np.allclose(
    orientations,
    orientations[0],
    atol=1e-4,
    rtol=0,
)

print("All slices have the same orientation:", same_orientation)

if not same_orientation:
    raise ValueError("Slice orientations differ; inspect before stacking.")

# Direction perpendicular to the images.
normal = np.cross(orientations[0, :3], orientations[0, 3:])
normal = normal / np.linalg.norm(normal)

# Project each image position onto that direction.
xyz = positions[["X_mm", "Y_mm", "Z_mm"]].to_numpy()

geometry_order = positions.copy()
geometry_order["Position_along_stack_mm"] = xyz @ normal

geometry_order = (
    geometry_order.sort_values("Position_along_stack_mm")
    .reset_index(drop=True)
)

print(
    "Perpendicular slice spacing (mm):",
    np.round(
        np.diff(geometry_order["Position_along_stack_mm"]),
        3,
    ),
)

display(
    geometry_order[["InstanceNumber", "Position_along_stack_mm"]]
)

All slices have the same orientation: True
Perpendicular slice spacing (mm): [4.635 4.635 4.635 4.635 4.635 4.635 4.635 4.635 4.635 4.635 4.635 4.635
 4.635 4.635 4.635 4.635 4.635 4.635 4.635 4.635 4.635]


,InstanceNumber,Position_along_stack_mm
0,1,61.062297
1,2,65.696846
2,3,70.331404
3,4,74.965954
4,5,79.600511
5,6,84.235061
6,7,88.869618
7,8,93.504168
8,9,98.138717
9,10,102.773274


# Overall Properties of the DICOM Data
| Property                    | Verified result               |
| --------------------------- | ----------------------------- |
| Files                       | 22                            |
| Image dimensions            | 512 × 512 pixels              |
| In-plane pixel spacing      | About 0.330 × 0.330 mm        |
| Slice thickness             | 3.4 mm                        |
| Perpendicular slice spacing | About 4.635 mm                |
| Orientation                 | Consistent across the series  |
| Physical ordering           | Matches `InstanceNumber` here |
